# Team Season - team_dashboard_by_shooting_splits

This notebook inspects the data **downloaded** from the `team_dashboard_by_shooting_splits` endpoint.

Source folder: `02_data/01_raw/2025_26/02_team_season/team_dashboard_by_shooting_splits`

Scope:
- Read parquet files in the folder
- Show how many files were downloaded and how many rows/columns they contain
- Report missing values (null cells) overall and by row/column distribution

Notes:
- Read-only analysis: no exports or writes


In [9]:
from pathlib import Path
import pandas as pd
import numpy as np

# Resolve project root by walking up to find 02_data
project_root = Path.cwd()
for _ in range(6):
    if (project_root / "02_data").exists():
        break
    project_root = project_root.parent

endpoint = "team_dashboard_by_shooting_splits"
data_dir = project_root / "02_data" / "01_raw" / "2025_26" / "02_team_season" / endpoint

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)


In [10]:
files = sorted(data_dir.glob("*.parquet"))
print(f"Endpoint: {endpoint}")
print(f"Folder: {data_dir}")
print(f"Parquet files: {len(files)}")

files_df = pd.DataFrame({
    "file": [f.name for f in files],
    "size_mb": [round(f.stat().st_size / 1e6, 3) for f in files],
})
files_df


Endpoint: team_dashboard_by_shooting_splits
Folder: /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/01_raw/2025_26/02_team_season/team_dashboard_by_shooting_splits
Parquet files: 30


,file,size_mb
0,team_dashboard_by_shooting_splits__team_id=161...,0.03
1,team_dashboard_by_shooting_splits__team_id=161...,0.03
2,team_dashboard_by_shooting_splits__team_id=161...,0.03
3,team_dashboard_by_shooting_splits__team_id=161...,0.03
4,team_dashboard_by_shooting_splits__team_id=161...,0.03
5,team_dashboard_by_shooting_splits__team_id=161...,0.03
6,team_dashboard_by_shooting_splits__team_id=161...,0.03
7,team_dashboard_by_shooting_splits__team_id=161...,0.03
8,team_dashboard_by_shooting_splits__team_id=161...,0.03
9,team_dashboard_by_shooting_splits__team_id=161...,0.03


In [11]:
dfs = [pd.read_parquet(f) for f in files]

per_file = pd.DataFrame({
    "file": [f.name for f in files],
    "rows": [len(df) for df in dfs],
    "cols": [df.shape[1] for df in dfs],
})
per_file


,file,rows,cols
0,team_dashboard_by_shooting_splits__team_id=161...,92,37
1,team_dashboard_by_shooting_splits__team_id=161...,89,37
2,team_dashboard_by_shooting_splits__team_id=161...,92,37
3,team_dashboard_by_shooting_splits__team_id=161...,89,37
4,team_dashboard_by_shooting_splits__team_id=161...,98,37
5,team_dashboard_by_shooting_splits__team_id=161...,93,37
6,team_dashboard_by_shooting_splits__team_id=161...,89,37
7,team_dashboard_by_shooting_splits__team_id=161...,90,37
8,team_dashboard_by_shooting_splits__team_id=161...,87,37
9,team_dashboard_by_shooting_splits__team_id=161...,91,37


In [12]:
df = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print(f"Combined shape: {df.shape}")
print(f"Total rows: {len(df)}")
print(f"Total columns: {df.shape[1] if len(df.columns) else 0}")


Combined shape: (2727, 37)
Total rows: 2727
Total columns: 37


In [13]:
rows, cols = df.shape
total_cells = rows * cols
null_cells = int(df.isna().sum().sum()) if total_cells else 0
null_pct = (null_cells / total_cells) if total_cells else 0

summary = pd.DataFrame({
    "rows": [rows],
    "cols": [cols],
    "total_cells": [total_cells],
    "null_cells": [null_cells],
    "null_pct": [round(null_pct * 100, 3)],
})
summary


,rows,cols,total_cells,null_cells,null_pct
0,2727,37,100899,4872,4.829


In [14]:
if df.empty:
    col_summary = pd.DataFrame()
else:
    col_nulls = df.isna().sum()
    col_null_pct = df.isna().mean()
    col_summary = (
        pd.DataFrame({
            "null_cells": col_nulls,
            "null_pct": (col_null_pct * 100).round(3),
        })
        .sort_values("null_pct", ascending=False)
    )

col_summary


,null_cells,null_pct
PLAYER_NAME,2145,78.658
PLAYER_ID,2145,78.658
GROUP_VALUE,582,21.342
PCT_AST_3PM_RANK,0,0.000
FG3A_RANK,0,0.000
FG3_PCT_RANK,0,0.000
EFG_PCT_RANK,0,0.000
BLKA_RANK,0,0.000
PCT_AST_2PM_RANK,0,0.000
PCT_UAST_2PM_RANK,0,0.000


In [15]:
if df.empty:
    row_dist = pd.DataFrame()
else:
    row_null_pct = df.isna().mean(axis=1)
    bins = [0, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 1.0]
    row_bins = pd.cut(row_null_pct, bins=bins, include_lowest=True)
    row_counts = row_bins.value_counts().sort_index()
    row_dist = pd.DataFrame({
        "rows": row_counts,
        "pct_rows": (row_counts / len(row_null_pct) * 100).round(3),
    })

row_dist


,rows,pct_rows
"(-0.001, 0.01]",0,0.000
"(0.01, 0.05]",582,21.342
"(0.05, 0.1]",2145,78.658
"(0.1, 0.25]",0,0.000
"(0.25, 0.5]",0,0.000
"(0.5, 0.75]",0,0.000
"(0.75, 1.0]",0,0.000


In [16]:
# Merge all teams, fill SEASON, split by TABLE_NAME, drop fully-null columns, report nulls, and export

files = sorted(data_dir.glob("*.parquet"))
dfs = [pd.read_parquet(f) for f in files]
df_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

if df_all.empty:
    print("No data to merge.")
else:
    # Fill SEASON using non-null value (typically from Overall rows)
    if "SEASON" in df_all.columns:
        season_value = df_all["SEASON"].dropna().unique()
        season_value = season_value[0] if len(season_value) else None
        if season_value is not None:
            df_all["SEASON"] = df_all["SEASON"].fillna(season_value)
            print(f"Filled SEASON with: {season_value}")
        else:
            print("SEASON is fully null; nothing to fill.")
    else:
        print("SEASON column not found.")

    table_col = "TABLE_NAME" if "TABLE_NAME" in df_all.columns else None
    if table_col is None:
        print("Required column not found: TABLE_NAME")
        print(df_all.columns)
    else:
        out_dir = project_root / "02_data" / "02_cleaned" / "2025_26" / "02_team_season"
        out_dir.mkdir(parents=True, exist_ok=True)

        table_values = sorted([v for v in df_all[table_col].dropna().unique()])

        null_summary = []
        for name in table_values:
            df_part = df_all.loc[df_all[table_col] == name].copy()

            if df_part.empty:
                print(f"\n{name}: no rows")
                fully_null = []
            else:
                col_nulls = df_part.isna().sum()
                col_null_pct = (df_part.isna().mean() * 100).round(3)
                cols_with_nulls = col_nulls[col_nulls > 0].index.tolist()
                fully_null = col_nulls[col_nulls == len(df_part)].index.tolist()

                print(f"\n{name}: columns with any nulls ({len(cols_with_nulls)})")
                if cols_with_nulls:
                    display(pd.DataFrame({
                        "null_cells": col_nulls[cols_with_nulls],
                        "null_pct": col_null_pct[cols_with_nulls],
                    }).sort_values("null_pct", ascending=False))

                print(f"{name}: fully null columns ({len(fully_null)})")
                if fully_null:
                    print(fully_null)

            # Drop fully-null columns (they belong to other tables)
            if fully_null:
                df_part = df_part.drop(columns=fully_null)

            # Recompute null summary AFTER dropping fully-null columns
            rows, cols = df_part.shape
            total_cells = rows * cols
            null_cells = int(df_part.isna().sum().sum()) if total_cells else 0
            null_pct = (null_cells / total_cells * 100) if total_cells else 0

            null_summary.append({
                "table_name": name,
                "rows": rows,
                "cols": cols,
                "total_cells": total_cells,
                "null_cells": null_cells,
                "null_pct": round(null_pct, 3),
            })

            out_path = out_dir / f"team_dashboard_by_shooting_splits__{name}.parquet"
            df_part.to_parquet(out_path, index=False)
            print(f"Saved {name}: {len(df_part)} rows -> {out_path}")

        null_summary_df = pd.DataFrame(null_summary).sort_values("table_name")
        print("\nNull summary by TABLE_NAME (after dropping fully-null columns):")
        print(null_summary_df)


Filled SEASON with: 2025-26

AssistedBy: columns with any nulls (1)


,null_cells,null_pct
GROUP_VALUE,582,100.0


AssistedBy: fully null columns (1)
['GROUP_VALUE']
Saved AssistedBy: 582 rows -> /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dashboard_by_shooting_splits__AssistedBy.parquet

AssitedShotTeamDashboard: columns with any nulls (2)


,null_cells,null_pct
PLAYER_ID,60,100.0
PLAYER_NAME,60,100.0


AssitedShotTeamDashboard: fully null columns (2)
['PLAYER_ID', 'PLAYER_NAME']
Saved AssitedShotTeamDashboard: 60 rows -> /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dashboard_by_shooting_splits__AssitedShotTeamDashboard.parquet

OverallTeamDashboard: columns with any nulls (2)


...

Shot8FTTeamDashboard: fully null columns (2)
['PLAYER_ID', 'PLAYER_NAME']
Saved Shot8FTTeamDashboard: 150 rows -> /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dashboard_by_shooting_splits__Shot8FTTeamDashboard.parquet

ShotAreaTeamDashboard: columns with any nulls (2)


,null_cells,null_pct
PLAYER_ID,210,100.0
PLAYER_NAME,210,100.0


ShotAreaTeamDashboard: fully null columns (2)
['PLAYER_ID', 'PLAYER_NAME']
Saved ShotAreaTeamDashboard: 210 rows -> /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dashboard_by_shooting_splits__ShotAreaTeamDashboard.parquet

ShotTypeTeamDashboard: columns with any nulls (2)


,null_cells,null_pct
PLAYER_ID,1425,100.0
PLAYER_NAME,1425,100.0


ShotTypeTeamDashboard: fully null columns (2)
['PLAYER_ID', 'PLAYER_NAME']
Saved ShotTypeTeamDashboard: 1425 rows -> /Users/pablodiazgonzalez/Documents/MachineLearning/AlgoBasket/02_data/02_cleaned/2025_26/02_team_season/team_dashboard_by_shooting_splits__ShotTypeTeamDashboard.parquet

Null summary by TABLE_NAME (after dropping fully-null columns):
                 table_name  rows  cols  total_cells  null_cells  null_pct
0                AssistedBy   582    36        20952           0       0.0
1  AssitedShotTeamDashboard    60    35         2100           0       0.0
2      OverallTeamDashboard    30    35         1050           0       0.0
3      Shot5FTTeamDashboard   270    35         9450           0       0.0
4      Shot8FTTeamDashboard   150    35         5250           0       0.0
5     ShotAreaTeamDashboard   210    35         7350           0       0.0
6     ShotTypeTeamDashboard  1425    35        49875           0       0.0
